In [ ]:
#@title 1. Install the classroom demo
%pip install -q "kaist-rl-lab[interactive]==0.1.19" "gradio==6.26.0"

# Coffee trajectories — instructor notebook

Run this notebook **in your personal Google account**. It creates a collection folder in your Drive and an upload-only endpoint for the class.
Students use their own Colab runtimes and personal accounts; they do not mount your Drive or receive edit access to it.

Run the install cell, then the setup cell. Approve Colab's Drive connection when prompted. A standard CPU runtime is sufficient.
The default destination is **My Drive / KAIST Coffee Trajectories**; you can change the path below to your preferred folder.
Distribute the resulting collector URL and lecture code with the student notebook. Keep this runtime connected throughout the lecture.


In [ ]:
#@title 2. Connect your Drive and start receiving trajectories
collection_folder = "/content/drive/MyDrive/KAIST Coffee Trajectories" #@param {type:"string"}

from google.colab import drive
import secrets
from kaist_rl_lab.apps.coffee_demonstrations import TrajectoryCollector, build_collector

drive.mount("/content/drive")
if "collector_app" in globals():
    collector_app.close()

lecture_code = secrets.token_urlsafe(16)
collector = TrajectoryCollector(collection_folder, lecture_code)
collector_app = build_collector(collector)
collector_app.launch(share=True, inline=False, debug=False, show_error=False)

print("Student collector URL:", collector_app.share_url)
print("Lecture code:", lecture_code)
print("Saving to:", collector.directory)
print("Give students the URL and lecture code above. Each .npz is one complete attempt.")


## Live lecture view

Run the next cell to refresh the received-attempt list every three seconds. Stop that cell to pause the display; the collector continues accepting submissions.
Only this instructor notebook can list the saved recordings. The public endpoint returns a receipt for the submitted attempt.
A student's repeated upload of the same attempt is counted once.


In [ ]:
#@title 3. Watch submissions arrive (optional)
import time
import pandas as pd
from IPython.display import clear_output, display

try:
    while True:
        rows = collector.summary()
        clear_output(wait=True)
        print(f"Received {len(rows)} attempts · {sum(r['transitions'] for r in rows)} transitions")
        display(pd.DataFrame(rows))
        time.sleep(3)
except KeyboardInterrupt:
    print("Display paused. The collector is still running.")


## Load the demonstrations for behavior cloning

Each archive contains `observations`, `actions`, `rewards`, `next_observations`, `terminated`, `truncated`, and JSON `metadata`.
Arrays load with `allow_pickle=False`. Metadata includes episode/participant codes, timing, the package and physics versions, ordered observation/joint names, target, fill, spill, and success.
Early submissions mark their final transition truncated. The records retain the environment's 0.125-second action interval and 1/64-second physics substeps.

Stop the monitor before running this cell. Use `episode_ids` to split whole attempts into training and validation sets, rather than mixing adjacent states from one attempt across both sets.


In [ ]:
#@title 4. Load the received observation/action pairs (optional)
from kaist_rl_lab.apps.coffee_demonstrations import load_behavior_cloning_data

data = load_behavior_cloning_data(collection_folder)
observations = data["observations"]  # (transitions, 16)
actions = data["actions"]            # (transitions, 6), commands in [-1, 1]
episode_ids = data["episode_ids"]
print("Observations:", observations.shape, "Actions:", actions.shape)


When class ends, run `collector_app.close()` or disconnect the runtime. The submitted files remain in your Drive.
If you rerun the setup cell, it generates a new lecture code; give students the new URL/code pair. Previously saved recordings stay in the collection folder.
Keep this instructor notebook private: the student notebook is the one to distribute.
